In [1]:
import matplotlib.pyplot as plt
import sys
import os
import importlib

# Add the parent directory to the path to import src as a package
sys.path.insert(0, os.path.abspath('..'))
from src import dataloader

importlib.reload(dataloader)
from src import utils
from src import export

importlib.reload(export)
from src.export import export_to_xarray



%matplotlib widget 
plot_flag = False

In [2]:
dyad_id = "W_010"
lowcut=1.0
highcut=40.0
eeg_filter_type = 'iir' # choose 'fir' or 'iir' for EEG filtering
q=8  # decimation factor
multimodal_data = dataloader.create_multimodal_data(data_base_path = "../data", 
                                                    dyad_id = dyad_id, 
                                                    load_eeg=True, 
                                                    load_et=False, 
                                                    lowcut=lowcut, 
                                                    highcut=highcut, 
                                                    eeg_filter_type=eeg_filter_type, 
                                                    interpolate_et_during_blinks_threshold=0.3,
                                                    median_filter_size=64,
                                                    low_pass_et_order=351,
                                                    et_pos_cutoff=128,
                                                    et_pupil_cutoff=4,
                                                    pupil_model_confidence=0.9,
                                                    decimate_factor=q,
                                                    plot_flag=plot_flag)

FileNotFoundError: [Errno 2] No such file or directory: '../data/W_010/eeg/W_010.obci.xml'

In [ ]:
 # seconds
#selected_event =  'Peppa' #'Incredibles' 'Peppa' 'Brave' 'Talk_1' 'Talk_2'
#member = 'ch' #'cg'

#selected_modality = 'EEG' # choose 'EEG', 'ECG', 'ET' , 'IBI' or 'diode' for modality to export to xarray (diode is the reference for checking the correctnes of event slicing)
#selected_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'M1', 'T3', 'C3', 'Cz', 'C4', 'T4', 'M2', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2']
#selected_modality = 'ET'
#selected_channels = ['x', 'y', 'pupil', 'blinks']

#selected_modality = 'ECG'
#selected_channels = ['ECG']

#selected_modality = 'IBI'
#selected_channels = ['IBI']
#selected_modality = 'diode'
#selected_channels = ['diode']
time_margin = 10
selected_channels = {'EEG': ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'M1', 'T3', 'C3', 'Cz', 'C4', 'T4', 'M2', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'],
                     'ET':['x', 'y', 'pupil', 'blinks'],
                     'ECG':['ECG'],
                     'IBI':['IBI']}
selected_modalities = multimodal_data.modalities
members = {'ch':'child', 'cg':'caregiver'}
path = os.path.join('UNIWAW_imported', str(multimodal_data.id))
if not os.path.exists(path):
    os.makedirs(path)
for modality in selected_modalities:
    path = os.path.join(path, modality)
    if not os.path.exists(path):
        os.makedirs(path)
    for who, member in members.items():
        path = os.path.join(path, member)
        if not os.path.exists(path):
            os.makedirs(path)
        for event in multimodal_data.events.get('name'):    
            data_xr = export_to_xarray(multimodal_data = multimodal_data, 
                                   selected_event=event, 
                                   selected_channels=selected_channels.get(modality), 
                                   selected_modality=modality, 
                                   member=who, 
                                   time_margin=time_margin)
            file_path = os.path.join(path, f'_W_{multimodal_data.id}_{modality}_{who}.nc')
            data_xr.to_netcdf(file_path, engine='netcdf4')




